# 🦙 08 — Ollama Deployment Test

**Tujuan:** Jalankan model GGUF via Ollama lokal, hitung ulang metrik untuk membandingkan dengan versi safetensors.

**Lokasi:** VS Code (panggil API Ollama lokal)

---
**Output:** `results/generations_log/ollama_harmful.csv` + append ke metrics.csv

Saved results are in `results/reference/`. Your results may differ with other inputs or settings. See `docs/results.md` for details.

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "src" / "config.py").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Open the notebook from inside the project directory.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [ ]:
import sys
sys.path.insert(0, str(PROJECT_ROOT))

import json
import requests
import pandas as pd

from src.config import GGUF_DIR, N_INST_TRAIN, N_INST_TEST, LOGS_DIR, RESULTS_DIR, SEED
from src.data_loader import get_harmful_instructions, get_harmless_instructions, create_holdout_set, get_train_test_split
from src.metrics import compute_refusal_rate, compute_asr, classify_refusal

from src.config import OLLAMA_HOST, OLLAMA_MODEL
OLLAMA_URL = f"{OLLAMA_HOST}/api/generate"
MODEL_NAME = OLLAMA_MODEL


In [ ]:
# --- Import model GGUF ke Ollama ---
# Jalankan INI SEKALI aja sebelum generate
# Pastikan file Modelfile ada di folder proyek
import subprocess
import sys
sys.path.insert(0, str(PROJECT_ROOT))
from pathlib import Path
from src.config import ROOT

MODEL_TAG = MODEL_NAME
MODELLFILE_PATH = ROOT / "Modelfile"

# Cek kalo model udah ada di Ollama
result = subprocess.run(
    ["ollama", "list"],
    capture_output=True, text=True
)
if MODEL_TAG in result.stdout:
    print(f"Model {MODEL_TAG} sudah terdaftar. Skip import.")
else:
    print(f"Importing {MODEL_TAG}...")
    result = subprocess.run(
        ["ollama", "create", MODEL_TAG, "-f", str(MODELLFILE_PATH)],
        cwd=str(ROOT),
    )
    if result.returncode == 0:
        print(f"Done! Model {MODEL_TAG} siap digunakan.")
    else:
        print(f"Error: {result.stderr}")


In [ ]:
# ─── Setup Ollama Model ───
# Prasyarat: Ollama terinstall, Modelfile dibuat, model di-import

print("Langkah prasyarat (jalankan di terminal):")
print(f"""
  1. Buka terminal
  2. cd {GGUF_DIR}
  3. echo FROM ./model.gguf > Modelfile
  4. ollama create {MODEL_NAME} -f Modelfile
  5. ollama run {MODEL_NAME}  (test manual)
""")

# Cek apakah Ollama jalan
try:
    r = requests.get(f"{OLLAMA_HOST}/api/tags", timeout=3)
    models = r.json().get('models', [])
    available = [m['name'] for m in models]
    print(f"Ollama models: {available}")
    if MODEL_NAME in available:
        print(f"Model '{MODEL_NAME}' tersedia!")
    else:
        print(f"Model '{MODEL_NAME}' belum di-import — import dulu.")
except Exception as e:
    print(f"Ollama tidak bisa diakses: {e}")

In [ ]:
# ─── Fungsi generate via Ollama API ───
def ollama_generate(prompt, model_name=MODEL_NAME, max_tokens=150):
    payload = {
        "model": model_name,
        "prompt": prompt,
        "stream": False,
        "options": {"num_predict": max_tokens}
    }
    r = requests.post(OLLAMA_URL, json=payload, timeout=60)
    r.raise_for_status()
    body = r.json()
    if "error" in body or "response" not in body:
        raise RuntimeError(f"Ollama generation failed: {body.get('error', 'missing response')}")
    return body["response"]


def batch_ollama(prompts, model_name=MODEL_NAME):
    return [ollama_generate(p, model_name) for p in prompts]

In [ ]:
# ─── Load data & generate ───
harmful = get_harmful_instructions()
harmless = get_harmless_instructions()
holdout_harmful, holdout_harmless = create_holdout_set(harmful, harmless)
_, _, harmful_test, harmless_test = get_train_test_split(harmful, harmless, holdout_harmful, holdout_harmless, n_train=N_INST_TRAIN, n_test=N_INST_TEST)
print(f"Test: {len(harmful_test)} harmful")

print("Generating via Ollama...")
ollama_gens = batch_ollama(harmful_test)

for i in range(min(2, len(ollama_gens))):
    refused = classify_refusal(ollama_gens[i])
    print(f"\n--- Ollama {i} (Refused: {refused}) ---")
    print(f"Q: {harmful_test[i]}")
    print(f"A: {ollama_gens[i][:200]}")

In [ ]:
# ─── Metrik & simpan ───
ollama_rr = compute_refusal_rate(ollama_gens)
ollama_asr = compute_asr(ollama_gens, harmful_test)

print(f"\nOLLAMA METRICS")
print(f"RR: {ollama_rr:.2%}")
print(f"ASR heuristic: {ollama_asr:.2%}")

# Simpan
df_gen = pd.DataFrame({
    'prompt': harmful_test,
    'generation': ollama_gens,
    'refused': [classify_refusal(g) for g in ollama_gens],
    'condition': 'ollama_abliterated'
})
df_gen.to_csv(LOGS_DIR / 'ollama_harmful.csv', index=False)

df_metrics = pd.DataFrame([{
    'condition': 'ollama_abliterated',
    'refusal_rate': ollama_rr,
    'asr_heuristic': ollama_asr,
    'n_samples': len(ollama_gens),
    'model': 'GGUF F16 via Ollama',
}])
df_metrics.to_csv(RESULTS_DIR / 'metrics.csv', index=False, mode='a', header=False)
print(f"Ollama metrics appended!")

Done. Compare the saved results in `results/reference/metrics.csv`, or read `docs/results.md` for the numbers and their limits.